# Ejercicio 03 · Corpus público y clasificador de texto reproducible

**Autor:** Edwin Ramón José Nolasco  
**Pregunta:** ¿cómo se compara TF-IDF con Complement Naive Bayes y regresión logística para distinguir SMS legítimos (ham) de spam sin compartir textos duplicados entre entrenamiento y prueba?

Esta notebook carga el ZIP público de UCI, audita calidad y duplicados, separa antes de ajustar el vocabulario, compara tres modelos en el mismo holdout y registra errores. La descarga incluida permite repetirla sin red.

Corpus: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), licencia CC BY 4.0. Para instalar y ejecutar desde cero, consulta `../README.md`.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from nlp_project import read_corpus, audit_data, clean_for_split, run_experiment
import pandas as pd


## 1. Auditoría del corpus

Se valida esquema, clases, ausencias y duplicados antes de entrenar.

In [2]:
raw = read_corpus()
audit = audit_data(raw)
print(f"Dimensiones: {raw.shape[0]:,} filas × {raw.shape[1]} columnas")
print(f"SHA-256 ZIP: {audit['sha256_zip']}")
print(f"Etiquetas: {audit['class_counts']}")
print(f"Nulos / textos vacíos: {audit['missing_required']} / {audit['empty_text']}")
print(f"Duplicados exactos: {audit['duplicate_rows_exact']}")
print(f"Duplicados normalizados: {audit['duplicate_rows_normalized']}")
print(f"Grupos con etiquetas en conflicto: {audit['conflicting_duplicate_groups']}")
print(f"Mediana de caracteres: {audit['message_length_median']:.0f}")


Dimensiones: 5,572 filas × 2 columnas
SHA-256 ZIP: 1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3
Etiquetas: {'ham': 4825, 'spam': 747}
Nulos / textos vacíos: 0 / 0
Duplicados exactos: 403
Duplicados normalizados: 415
Grupos con etiquetas en conflicto: 0
Mediana de caracteres: 62


## 2. Prevención de fuga y evaluación

Se excluyen grupos con etiquetas en conflicto y se conserva una instancia por texto normalizado. Luego se hace una partición estratificada 75/25, semilla 42. Los vectorizadores TF-IDF viven dentro de cada pipeline: el vocabulario se aprende solo con entrenamiento.

In [3]:
clean, cleaning = clean_for_split(raw)
print(f"Filas retiradas por conflictos: {cleaning['conflicting_rows_removed']}")
print(f"Duplicados retirados antes del split: {cleaning['duplicate_rows_removed']}")
print(f"Textos únicos: {len(clean):,}")
result = run_experiment(raw)
print(f"Train: {result['n_train']:,}; prueba: {result['n_test']:,}; semilla: 42")
print(f"Modelo seleccionado por F1 macro: {result['selected']}")
print("\nComparación (mismo split):")
print(result['scores'].to_string(index=False, float_format=lambda x: f"{x:.6f}"))


Filas retiradas por conflictos: 0
Duplicados retirados antes del split: 415
Textos únicos: 5,157
Train: 3,867; prueba: 1,290; semilla: 42
Modelo seleccionado por F1 macro: logistic_regression

Comparación (mismo split):
              model  f1_macro  accuracy
logistic_regression  0.959727  0.982171
      complement_nb  0.953372  0.979845
              dummy  0.466722  0.875194


## 3. Métricas, clases y matriz

Las métricas por clase exponen el rendimiento de spam frente al de ham. La matriz usa filas = clase real y columnas = clase predicha.

In [4]:
print(result['class_metrics'].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("Matriz de confusión [ham, spam]:")
print(result['confusion'])
print(f"Errores del modelo seleccionado en el holdout: {len(result['errors'])}")


label  precision  recall     f1  support
  ham     0.9920  0.9876 0.9898     1129
 spam     0.9157  0.9441 0.9297      161
Matriz de confusión [ham, spam]:
[[1115   14]
 [   9  152]]
Errores del modelo seleccionado en el holdout: 20


### Perfil del corpus

![Distribución de clases y longitudes](../reports/corpus_profile.png)

### Matriz de confusión

![Matriz de confusión](../reports/confusion_matrix.png)


## 4. Revisión de 20 errores

Son 20 predicciones incorrectas reales del holdout. Las categorías y notas son un borrador asistido por IA, no una validación del estudiante. Revise el contexto, corrija las etiquetas de análisis si hace falta y explique los patrones antes de entregar.

In [5]:
errors = pd.read_csv(ROOT / "reports/error_analysis.csv")
print(errors[["text", "real", "predicted", "category_suggested", "analysis_suggested"]]
      .to_string(index=False, max_colwidth=100))


                                                                                                text real predicted            category_suggested                                                                                   analysis_suggested
                                                                   Yes.. now only saw your message..  ham      spam      Respuesta personal breve                     Pocas palabras y sin contexto; el mensaje no tiene señales promocionales claras.
                              Latest News! Police station toilet stolen, cops have nothing to go on! spam       ham           Titular humorístico             La forma de titular y el juego de palabras pueden parecer una oferta; requiere contexto.
                                                                    Are you free now?can i call now?  ham      spam             Ambigüedad léxica           Free significa disponible en conversación personal, aunque también aparece en promociones.
I got a call

## 5. Interpretación y límites

En esta ejecución, regresión logística TF-IDF obtuvo el F1 macro más alto (0.959727), por encima de Complement Naive Bayes (0.953372); el baseline mayoritario obtuvo 0.466722. El recall de spam fue 0.9441: 9 de 161 mensajes spam se clasificaron como ham. Es una comparación en una partición del corpus histórico en inglés, no una estimación para mensajes actuales o español dominicano. La ficha, tabla de errores, métricas y auditoría reproducible se guardan en `reports/` y `docs/`.